In [ ]:
import os
model_to_test = 'jettag-qkeras'
model_revision = 'Qkeras_Pruned'
hls4ml_revision = 'VU'

base_dir = os.path.abspath(model_to_test)
model_dir = os.path.join(base_dir, model_revision)
os.makedirs(model_dir, exist_ok=True)

description = f"""
# Model Configuration

Aim is to run inference on HW (VitisUnified with custom 2025-script-patch)
Problems running HGQ2-models; Vitis Unified sets io_stream, but HGQ2 requires io_parallel for heteregenous activation. 
This is just to test different models.

- **Model architecture description**: {model_to_test}
- **Model Revision**: {model_revision}
- **HLS4ML Revision**: {hls4ml_revision}
- **Target Device**: KV260 (xck26-sfvc784-2LV-c)
- **Dataset**: HLS4ML LHC Jets
- **Vivado/Vitis**: 2025.2
"""
output_dir = os.path.join(model_dir, f"hls4ml_prj_{hls4ml_revision}")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
    f.write(description)

In [2]:
import sys
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tf_keras as keras
sys.modules["tensorflow.keras"] = keras
sys.modules["keras"] = keras

import keras_tuner as kt
import qkeras
from qkeras import QDense, QConv2D, QActivation
from qkeras.quantizers import quantized_bits, quantized_relu

import tensorflow as tf
from tf_keras.models import Sequential
from tf_keras.layers import MaxPooling2D, Activation, Flatten, Dropout
import numpy as np

2026-05-06 13:30:39.416798: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778067039.437899   35400 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778067039.443411   35400 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-06 13:30:39.467643: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
X_train_val = np.load('Data/x_train_val.npy')
X_test = np.load('Data/x_test.npy')
y_train_val = np.load('Data/y_train_val.npy')
y_test = np.load('Data/y_test.npy')
classes = np.load('Data/classes.npy', allow_pickle=True)

In [ ]:
inputs = keras.Input(shape=(16,), name='input_layer')

x = QDense(
    64,
    name='qdense0',
    kernel_quantizer=quantized_bits(2, 0, alpha=1),
    bias_quantizer=quantized_bits(2, 0, alpha=1),
    kernel_initializer='lecun_uniform',
    kernel_regularizer=keras.regularizers.l1(1e-4)
)(inputs)
x = QActivation(activation=quantized_relu(2), name='relu0')(x)

x = QDense(
    32,
    name='qdense1',
    kernel_quantizer=quantized_bits(2, 0, alpha=1),
    bias_quantizer=quantized_bits(2, 0, alpha=1),
    kernel_initializer='lecun_uniform',
    kernel_regularizer=keras.regularizers.l1(1e-4)
)(x)
x = QActivation(activation=quantized_relu(2), name='relu1')(x)

x = QDense(
    32,
    name='qdense2',
    kernel_quantizer=quantized_bits(2, 0, alpha=1),
    bias_quantizer=quantized_bits(2, 0, alpha=1),
    kernel_initializer='lecun_uniform',
    kernel_regularizer=keras.regularizers.l1(1e-4)
)(x)
x = QActivation(activation=quantized_relu(2), name='relu2')(x)

x = QDense(
    5,
    name='qdense3',
    kernel_quantizer=quantized_bits(2, 0, alpha=1),
    bias_quantizer=quantized_bits(2, 0, alpha=1),
    kernel_initializer='lecun_uniform',
    kernel_regularizer=keras.regularizers.l1(1e-4)
)(x)

outputs = keras.layers.Activation(activation='softmax', name='softmax')(x)

model = keras.Model(inputs=inputs, outputs=outputs)


I0000 00:00:1778067044.588810   35400 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
/home/slopin/DAT255-project/.venv_qkeras2/lib/python3.10/site-packages/tf_keras/src/initializers/initializers.py:121: UserWarning: The initializer LecunUniform is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


In [5]:
model.summary()

# Save the model summary to a text file
with open(os.path.join(model_dir, "summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_layer (InputLayer)    [(None, 16)]              0         
                                                                 
 qdense0 (QDense)            (None, 64)                1088      
                                                                 
 relu0 (QActivation)         (None, 64)                0         
                                                                 
 qdense1 (QDense)            (None, 32)                2080      
                                                                 
 relu1 (QActivation)         (None, 32)                0         
                                                                 
 qdense2 (QDense)            (None, 32)                1056      
                                                                 
 relu2 (QActivation)         (None, 32)                0     

In [ ]:
import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.sparsity import keras as sparsity

#Calculate endstep based on the dataset size and batch size
batch_size = 1024
epochs = 10
num_samples = len(X_train_val) * 0.75 
steps_per_epoch = np.ceil(num_samples / batch_size).astype(np.int32)
end_step = steps_per_epoch * epochs

pruning_params = {
    'pruning_schedule': sparsity.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.75, # Target 75% sparsity
        begin_step=0,
        end_step=end_step
    )
}
model = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

callbacks = [
    #using EarlyStopping for the restore_best_weight function isntead of ModelCheckpoint, 
    #though we dont want to actually stop before the end epoch due to the pruning setup.
    keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.2, patience=2, min_lr=1e-6, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
    
    #pruning specific
    sparsity.UpdatePruningStep(), 
    sparsity.PruningSummaries(log_dir='model_pruned/logs')
]

In [ ]:
opt = keras.optimizers.Adam(learning_rate=5e-3)
model.compile(opt, loss=['categorical_crossentropy'], metrics=['accuracy'])

In [8]:
model.fit(
    X_train_val, y_train_val,
    validation_data=(X_test, y_test),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=callbacks,
    shuffle=True
)

model = sparsity.strip_pruning(model)

/home/slopin/DAT255-project/.venv_qkeras2/lib/python3.10/site-packages/tf_keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(


Epoch 1/10


I0000 00:00:1778067067.094348   35535 service.cc:148] XLA service 0x763b30107e00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778067067.094425   35535 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-05-06 13:31:07.101926: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778067067.117468   35535 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1778067067.209143   35535 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


649/649 [==============================] - 22s 27ms/step - loss: 1.0281 - accuracy: 0.6719 - val_loss: 0.9577 - val_accuracy: 0.6980 - lr: 0.0050
Epoch 2/10
649/649 [==============================] - 16s 25ms/step - loss: 0.9296 - accuracy: 0.7026 - val_loss: 0.9133 - val_accuracy: 0.7076 - lr: 0.0050
Epoch 3/10
649/649 [==============================] - 15s 23ms/step - loss: 0.8935 - accuracy: 0.7119 - val_loss: 0.8858 - val_accuracy: 0.7133 - lr: 0.0050
Epoch 4/10
649/649 [==============================] - 15s 24ms/step - loss: 0.8761 - accuracy: 0.7165 - val_loss: 0.8900 - val_accuracy: 0.6913 - lr: 0.0050
Epoch 5/10
649/649 [==============================] - 14s 22ms/step - loss: 0.8714 - accuracy: 0.7169 - val_loss: 0.8762 - val_accuracy: 0.7170 - lr: 0.0050
Epoch 6/10
649/649 [==============================] - 15s 24ms/step - loss: 0.8747 - accuracy: 0.7167 - val_loss: 0.8606 - val_accuracy: 0.7260 - lr: 0.0050
Epoch 7/10
649/649 [==============================] - 14s 22ms/step -

In [10]:
#Has to be compiled again after stripping the pruning wrappers
model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=["accuracy"]) 

score = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])
test_acc = score[1]

model_name=f"model_{model_revision}_acc={test_acc:.4f}.keras"
model.save(os.path.join(model_dir, model_name))

Test loss: 0.8663215637207031
Test accuracy: 0.7279096245765686


/home/slopin/DAT255-project/.venv_qkeras2/lib/python3.10/site-packages/tf_keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(
